# Bathymetry

Bathymetry (`Topo`) sits between the horizontal grid and the vertical grid: it needs a
`Grid` to attach to (see [Grids](grids.ipynb)), and its `max_depth` typically feeds the
vertical grid (also in [Grids](grids.ipynb)).

CrocoDash re-exports `Topo` unmodified from [mom6_forge](https://github.com/NCAR/mom6_forge)
— the full API and editing mechanics live in the
[mom6_forge documentation](https://ncar.github.io/mom6_forge/), particularly the
bathymetry notebooks (`3_custom_bathy`, `4_ingest_landmask`, `5_modify_existing`,
`6_coarsen_existing`, `10_cressman_interpolation`) and the Topo Editor widget
walkthrough (`6_demo_editors`, Section 2).

This notebook covers:
- [Section 1](#section-1-remap-an-existing-bathymetry-dataset) — remap an existing dataset (e.g. GEBCO) onto your grid
- [Section 2](#section-2-interactive-editing-topoeditor) — interactive editing with `TopoEditor`
- [Section 3](#section-3-pre-generated-or-flat-bathymetry) — load a pre-existing topo file, or build a flat/synthetic one

📖 [CrocoDash grids docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html) · 📖 [mom6_forge docs](https://ncar.github.io/mom6_forge/)

## Section 1: Remap an Existing Bathymetry Dataset

This needs a `Grid` to attach the bathymetry to — see [Grids](grids.ipynb) for how to
build one. Here we build a small throwaway grid so this notebook runs on its own.

In [ ]:
from CrocoDash.grid import Grid

grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 278.0, # min longitude in [0, 360]
  lenx = 3.0, # longitude extent in degrees
  ystart = 7.0, # min latitude in [-90, 90]
  leny = 3.0, # latitude extent in degrees
  name = "panama1",
)

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

In [ ]:
from pathlib import Path

bathymetry_path= Path("<GEBCO>")
# The GEBCO bathymetry path on Derecho is: <GEBCO>

if not bathymetry_path.exists():
    raise FileNotFoundError("Bathymetry file not found, please replace with path to bathymetry file")

topo.set_from_dataset(
    bathymetry_path = bathymetry_path,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

In [ ]:
topo.depth.plot()

## Section 2: Interactive Editing (`TopoEditor`)

`TopoEditor` is an ipywidgets-based bathymetry editor with undo/redo, version-controlled
edits (a per-domain `TopoLibrary/` directory records every change as a git-style
operation), and live plotting — useful for fixing isolated basins, carving channels, or
smoothing coastlines before case creation.

In [ ]:
%matplotlib ipympl
from CrocoDash.topo_editor import TopoEditor
TopoEditor(topo)

## Section 3: Pre-Generated or Flat Bathymetry

Two more ways to get a `Topo` without remapping a dataset:
- **[Section 3.1](#section-31-load-a-pre-generated-topo-file)** — load a topo file that already matches your grid.
- **[Section 3.2](#section-32-flat-bathymetry)** — a constant-depth ocean, useful for idealized/test cases.

### Section 3.1: Load a Pre-Generated Topo File

Use this when you already have a topography file that matches your grid — e.g. from a
previous run or a shared community grid like NWA12.

In [ ]:
from CrocoDash.grid import Grid
from CrocoDash.topo import Topo

grid = Grid.from_supergrid("<NWA_HGRID>")

bathymetry_path = "<NWA_BATHY>"

topo = Topo.from_topo_file(
    grid=grid,
    topo_file_path=bathymetry_path,
    min_depth=5,
)

In [ ]:
topo.depth.plot()

### Section 3.2: Flat Bathymetry

For idealized or test domains, skip remapping entirely and set a constant depth with
`topo.set_flat(depth)` — used throughout this gallery's synthetic-domain examples (e.g.
[Interior OBC Segments](advanced/interior_obc_segments.ipynb)).

In [ ]:
flat_topo = Topo(grid, min_depth=10.0, git=False)
flat_topo.set_flat(1000.0)

## Next steps

Once you have a `Grid`, `Topo`, and `VGrid` (see [Grids](grids.ipynb)), pass them to
`Case` — see [Case Setup](case_setup.ipynb).